In [1]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Set working directory
os.chdir("/mnt/nfs/CX000008_DS1/projects/btanasa/hypoxia/test_geneformer_5_full_h5ad_aug20")
print("Files:", os.listdir('.'))

Files: ['hypoxia_anndata_annotated.adata5_geneformer.h5ad', 'figs', 'hypoxia_anndata_annotated.h5ad', 'hypoxia_anndata_annotated.cells_metadata.csv', 'hypoxia_anndata_annotated.genes_metadata.csv', 'hypoxia_anndata_annotated.adata5_geneformer.loom']


In [2]:
import sys

# ==============================
# Environment setup (if needed)
# ==============================
env_path = "/home/btanasa/miniconda3/envs/torch-cu"
os.environ["CONDA_PREFIX"] = env_path
os.environ["CONDA_DEFAULT_ENV"] = "torch-cu"
os.environ["RETICULATE_PYTHON"] = f"{env_path}/bin/python"
print("✅ Environment set to torch-cu")
print("Python executable:", sys.executable)

✅ Environment set to torch-cu
Python executable: /home/btanasa/miniconda3/envs/torch-cu/bin/python


In [3]:
import os
import sys

print("✅ Environment set to torch_cu")
print("Python executable:", sys.executable)
print("CONDA_PREFIX:", os.environ.get("CONDA_PREFIX"))
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV"))

# Verify the setup
try:
    import torch
    print("✅ PyTorch available:", torch.__version__)
    print("✅ CUDA available:", torch.cuda.is_available())
except ImportError as e:
    print("❌ PyTorch import failed:", e)

try:
    from geneformer import TranscriptomeTokenizer
    print("✅ Geneformer available")
except ImportError as e:
    print("❌ Geneformer import failed:", e)

✅ Environment set to torch_cu
Python executable: /home/btanasa/miniconda3/envs/torch-cu/bin/python
CONDA_PREFIX: /home/btanasa/miniconda3/envs/torch-cu
CONDA_DEFAULT_ENV: torch-cu
✅ PyTorch available: 2.5.1
✅ CUDA available: True
✅ Geneformer available


In [10]:
# =====================================
# Paths and Directories
# =====================================
import os
from geneformer import TranscriptomeTokenizer

# Base directories
workdir   = "/mnt/nfs/CX000008_DS1/projects/btanasa/hypoxia/"
gf_dir    = "/mnt/nfs/CX000008_DS1/projects/btanasa/Geneformer/geneformer"  # Geneformer dictionaries

# Input/output directories
h5ad_input_dir = os.path.join(workdir, "test_geneformer_5_full_h5ad_aug20_input")
out_dir        = os.path.join(workdir, "test_geneformer_5_full_h5ad_aug20_dataset")
out_prefix     = "hypoxia_all_samples_tokens"

# Dictionary files
gene_median_file     = os.path.join(gf_dir, "gene_median_dictionary_gc104M.pkl")
token_dictionary_file = os.path.join(gf_dir, "token_dictionary_gc104M.pkl")
gene_mapping_file     = os.path.join(gf_dir, "ensembl_mapping_dict_gc104M.pkl")

# Sanity check
print("Working dir:", workdir)
print("Input h5ad dir:", h5ad_input_dir)
print("Output dir:", out_dir)
print("Dictionary files:")
for f in [gene_median_file, token_dictionary_file, gene_mapping_file]:
    print(" -", f, "✓" if os.path.exists(f) else "❌ MISSING")

Working dir: /mnt/nfs/CX000008_DS1/projects/btanasa/hypoxia/
Input h5ad dir: /mnt/nfs/CX000008_DS1/projects/btanasa/hypoxia/test_geneformer_5_full_h5ad_aug20_input
Output dir: /mnt/nfs/CX000008_DS1/projects/btanasa/hypoxia/test_geneformer_5_full_h5ad_aug20_dataset
Dictionary files:
 - /mnt/nfs/CX000008_DS1/projects/btanasa/Geneformer/geneformer/gene_median_dictionary_gc104M.pkl ✓
 - /mnt/nfs/CX000008_DS1/projects/btanasa/Geneformer/geneformer/token_dictionary_gc104M.pkl ✓
 - /mnt/nfs/CX000008_DS1/projects/btanasa/Geneformer/geneformer/ensembl_mapping_dict_gc104M.pkl ✓


In [4]:
# =====================================
# Load and inspect h5ad
# =====================================

h5ad_file="hypoxia_anndata_annotated.adata5_geneformer.h5ad"

prefix = Path(h5ad_file).stem
adata = sc.read_h5ad(h5ad_file)
print(f"✅ Loaded: {adata.n_obs} cells × {adata.n_vars} genes")

print("\n.var columns:", list(adata.var.columns))
print(adata.var.head())

print("\n.obs columns:", list(adata.obs.columns))
print(adata.obs.head())
print(f"\nCells: {adata.n_obs}")

print("\nCurrent directory:", os.getcwd())
print("Files in directory:")
for f in os.listdir():
    print(" -", f)

✅ Loaded: 104438 cells × 21862 genes

.var columns: ['mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'gene_name', 'is_ENST', 'is_ENSG', 'is_ENSP', 'gene_no_version', 'ensembl_id', 'gene_symbol']
                    mt   ribo     hb  n_cells_by_counts  mean_counts  \
ENSG00000243485  False  False  False                 15     0.000104   
ENSG00000237491  False  False  False               9948     0.085519   
ENSG00000177757  False  False  False                202     0.001402   
ENSG00000228794  False  False  False               8628     0.072111   
ENSG00000225880  False  False  False               3737     0.027376   

                 log1p_mean_counts  pct_dropout_by_counts  total_counts  \
ENSG00000243485           0.000104              99.989638            15

In [5]:
# Add cell_id column equal to the rownames (barcodes) of .obs
# adata.obs["cell_id"] = adata.obs.index.astype(str)

# Quick check
# print(adata.obs.head()[["cell_id"]])

# write the updated AnnData into the tokenizer INPUT directory
# updated_h5ad = os.path.join(h5ad_input_dir, "input_with_cell_id.h5ad")
# adata.write(updated_h5ad)
# print("✅ Wrote updated h5ad for tokenization:", updated_h5ad)

In [6]:
print("\n.obs columns:", list(adata.obs.columns))
print(adata.obs.head(5))
print(f"\nCells: {adata.n_obs}")


.obs columns: ['sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'condition', '_scvi_batch', '_scvi_labels', 'leiden', 'merged_labels', 'class', 'broad_class', 'n_counts', 'counts', 'cell_id']
                       sample  n_genes_by_counts  log1p_n_genes_by_counts  \
TCAAGACCACAAAGCG-1  AP4765_03              12290                 9.416623   
CTCCACATCGCCACTT-1  AP4765_03              11499                 9.350102   
CTTAGGATCTTTCCAA-1  AP4765_03              10365                 9.246286   
AATTTCCCATTGCTTT-1  AP4765_03              10081                 9.218507   
GATGACTTCTATCC

In [8]:
cols = ["class", "broad_class", "samples", "condition"]

for col in cols:
    if col in adata.obs.columns:
        print(f"\n=== {col} ===")
        uniques = sorted(map(str, adata.obs[col].dropna().unique()))
        print(f"Unique ({len(uniques)}): {uniques}")
        print("\nCounts:")
        print(adata.obs[col].value_counts(dropna=False))
    else:
        print(f"\n=== {col} ===")
        print("Column not found in adata.obs")


=== class ===
Unique (26): ['Astroglia', 'Choroid', 'Endo', 'Ependymal', 'Fibroblast-Like', 'Imm-CGE', 'Imm-LGE', 'Imm-MGE', 'Imm-MGE-Str', 'Inh-IPC', 'Inh-Injured', 'LGE-OB', 'Lymphoid', 'Medium-Spiny-Neuron-GABA', 'Medium-Spiny-Neuron_?', 'Microglia', 'Mixed_Inh', 'Myelinating-Olig', 'OPC', 'PTLH+ Str-IN', 'Premyelinating-Olig', 'VIM+', 'activated-NSCs', 'excitatory', 'outer-RG', 'quiescent_NSCs']

Counts:
class
Astroglia                   26548
Imm-MGE                     11355
Inh-Injured                 10689
Medium-Spiny-Neuron_?        9268
activated-NSCs               7465
Myelinating-Olig             7044
OPC                          6505
Imm-CGE                      6203
Microglia                    4358
Imm-LGE                      2559
Endo                         2134
Premyelinating-Olig          1908
excitatory                   1855
Medium-Spiny-Neuron-GABA     1497
Inh-IPC                       802
LGE-OB                        801
Mixed_Inh                     711
VIM

In [11]:
# =====================================
# Tokenizer (Geneformer V2 ready)
# =====================================
tokenizer = TranscriptomeTokenizer(
    custom_attr_name_dict={
        "cell_id": "obs_names",  # 👈 This preserves cell barcodes from AnnData.obs.index
        "leiden": "leiden",
        "class": "class",
        "condition": "condition",
        "broad_class": "broad_class",
    },
    gene_median_file = gene_median_file,
    token_dictionary_file = token_dictionary_file,
    gene_mapping_file = gene_mapping_file,
    collapse_gene_ids = True,
    model_input_size = 4096,
    special_token = True,
    model_version = "V2",
)

# Run tokenization
tokenizer.tokenize_data(
    data_directory = h5ad_input_dir,
    output_directory = out_dir,
    output_prefix = out_prefix,
    file_format = "h5ad"
)

print("✅ Tokenization complete:", out_prefix)

Tokenizing /mnt/nfs/CX000008_DS1/projects/btanasa/hypoxia/test_geneformer_5_full_h5ad_aug20_input/hypoxia_anndata_annotated.adata5_geneformer.h5ad


/home/btanasa/miniconda3/envs/torch-cu/lib/python3.12/site-packages/geneformer/tokenizer.py:540: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/home/btanasa/miniconda3/envs/torch-cu/lib/python3.12/site-packages/geneformer/tokenizer.py:543: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


/mnt/nfs/CX000008_DS1/projects/btanasa/hypoxia/test_geneformer_5_full_h5ad_aug20_input/hypoxia_anndata_annotated.adata5_geneformer.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
✅ Tokenization complete: hypoxia_all_samples_tokens


In [12]:
import os, glob

print("Input dir exists:", os.path.isdir(h5ad_input_dir))
print("Input dir contents:", os.listdir(h5ad_input_dir) if os.path.isdir(h5ad_input_dir) else "N/A")

print("Out dir exists:", os.path.isdir(out_dir))
print("Out dir contents:", os.listdir(out_dir) if os.path.isdir(out_dir) else "N/A")

# Look for any *.dataset subfolders right under out_dir
candidates = [p for p in glob.glob(os.path.join(out_dir, "*.dataset")) if os.path.isdir(p)]
print("Direct .dataset candidates:", candidates)


Input dir exists: True
Input dir contents: ['hypoxia_anndata_annotated.adata5_geneformer.h5ad']
Out dir exists: True
Out dir contents: ['hypoxia_all_samples_tokens.dataset']
Direct .dataset candidates: ['/mnt/nfs/CX000008_DS1/projects/btanasa/hypoxia/test_geneformer_5_full_h5ad_aug20_dataset/hypoxia_all_samples_tokens.dataset']


In [13]:
# =====================================
# Load tokenized dataset (robust: auto-detect .dataset folder)
# =====================================
from datasets import load_from_disk
import os
import glob  # Add this import

# Preferred path if naming matches:
preferred_dataset_dir = os.path.join(out_dir, f"{out_prefix}.dataset")

def find_dataset_dir(base_dir, preferred=None):
    # 1) Use preferred if it exists
    if preferred and os.path.isdir(preferred):
        return preferred
    # 2) Otherwise, search for any "*.dataset" folder with HF signature files
    candidates = [p for p in glob.glob(os.path.join(base_dir, "*.dataset")) if os.path.isdir(p)]
    for cand in candidates:
        files = set(os.listdir(cand))
        if {"dataset_info.json", "state.json"} <= files:
            return cand
    return None

dataset_dir = find_dataset_dir(out_dir, preferred_dataset_dir)
print("Looking for dataset at:", preferred_dataset_dir)
if dataset_dir is None:
    # Give helpful diagnostics and stop early
    print("Contents of out_dir:", os.listdir(out_dir))
    raise FileNotFoundError(
        f"No HuggingFace dataset folder found under: {out_dir}\n"
        f"Expected something like: {out_prefix}.dataset with dataset_info.json and state.json"
    )

print("Found dataset at:", dataset_dir)
token_data = load_from_disk(dataset_dir)

Looking for dataset at: /mnt/nfs/CX000008_DS1/projects/btanasa/hypoxia/test_geneformer_5_full_h5ad_aug20_dataset/hypoxia_all_samples_tokens.dataset
Found dataset at: /mnt/nfs/CX000008_DS1/projects/btanasa/hypoxia/test_geneformer_5_full_h5ad_aug20_dataset/hypoxia_all_samples_tokens.dataset


In [14]:
print("\n• Dataset structure:")
print(f"Type: {type(token_data)}")
print(f"Features: {token_data.features}")
print(f"• Number of tokenized samples: {len(token_data)}")

# Show a few samples (guard in case dataset is small)
def safe_show(ds, i, label=None):
    if i < len(ds):
        print(f"\n• {label or f'Sample #{i}'}:", ds[i])

safe_show(token_data, 0, "First sample")
safe_show(token_data, 1, "Second sample")
safe_show(token_data, 2, "Third sample")
safe_show(token_data, 2697, "Sample #2698")
safe_show(token_data, 2698, "Sample #2699")


• Dataset structure:
Type: <class 'datasets.arrow_dataset.Dataset'>
Features: {'input_ids': List(Value('int32')), 'obs_names': Value('string'), 'leiden': Value('string'), 'class': Value('string'), 'condition': Value('string'), 'broad_class': Value('string'), 'length': Value('int64')}
• Number of tokenized samples: 104438

• First sample: {'input_ids': [2, 15927, 16559, 6428, 173, 16581, 11382, 5253, 17462, 11346, 13547, 4822, 12702, 6676, 14646, 16091, 11106, 1183, 16422, 12281, 13467, 11702, 9824, 12895, 9752, 6796, 9426, 9948, 6121, 9578, 13452, 11301, 19888, 10669, 1808, 9560, 3169, 1504, 382, 3918, 1748, 12052, 15669, 14382, 17789, 9714, 9995, 7366, 9419, 8499, 14916, 3772, 15166, 2822, 4275, 16990, 3374, 2932, 15494, 8404, 7965, 8525, 9557, 10985, 2377, 9701, 13109, 5377, 9731, 9694, 19304, 3187, 15613, 6926, 6469, 11328, 15239, 4838, 4096, 12116, 9288, 4390, 5077, 9787, 4227, 7801, 7617, 11697, 3731, 557, 9596, 3104, 8015, 10560, 803, 11023, 10713, 10275, 2805, 9208, 12199, 9360

In [16]:
# =====================================
# Convert tokenized dataset → Pandas
# =====================================
import pandas as pd

# Inspect columns
print("Available columns:", token_data.column_names)

# Example: inspect one item
i = 0
item = token_data[i]
for k in ["cell_id", "leiden", "class", "broad_class", "condition"]:
    if k in item:
        print(f"{k}:", item[k])

# Select metadata columns into a DataFrame
meta_cols = ["cell_id", "leiden", "class", "broad_class", "condition"]
meta_cols = [c for c in meta_cols if c in token_data.column_names]

df_meta = token_data.select_columns(meta_cols).to_pandas()
print("\nMetadata head:\n", df_meta.head())
print("Rows:", len(df_meta))

# Convert full dataset (⚠ may be large!)
df_all = token_data.to_pandas()
print("\nFull DataFrame head:\n", df_all.head())
print("Rows:", len(df_all))

# =====================================
# Save to external files
# =====================================
prefix = out_prefix if "out_prefix" in locals() else "token_data"

csv_path = f"{prefix}.all.csv"
txt_path = f"{prefix}.preview.txt"

df_all.to_csv(csv_path, index=False)

with open(txt_path, "w", encoding="utf-8") as f:
    f.write(df_all.head().to_string())
    f.write(f"\n\nrows: {len(df_all)}\n")

print("✅ Saved CSV:", csv_path)
print("✅ Saved preview TXT:", txt_path)


Available columns: ['input_ids', 'obs_names', 'leiden', 'class', 'condition', 'broad_class', 'length']
leiden: 30
class: Inh-Injured
broad_class: Non-Neuronal
condition: 2mo_HIE

Metadata head:
   leiden                  class   broad_class condition
0     30            Inh-Injured  Non-Neuronal   2mo_HIE
1     21  Medium-Spiny-Neuron_?  Non-Neuronal   2mo_HIE
2     21  Medium-Spiny-Neuron_?  Non-Neuronal   2mo_HIE
3     17  Medium-Spiny-Neuron_?  Non-Neuronal   2mo_HIE
4     21  Medium-Spiny-Neuron_?  Non-Neuronal   2mo_HIE
Rows: 104438

Full DataFrame head:
                                            input_ids           obs_names  \
0  [2, 15927, 16559, 6428, 173, 16581, 11382, 525...  TCAAGACCACAAAGCG-1   
1  [2, 15143, 14526, 2480, 13184, 5960, 2682, 103...  CTCCACATCGCCACTT-1   
2  [2, 5960, 15143, 3919, 1363, 14526, 14850, 131...  CTTAGGATCTTTCCAA-1   
3  [2, 9065, 14526, 13184, 3919, 7988, 14850, 195...  AATTTCCCATTGCTTT-1   
4  [2, 14468, 3919, 15143, 1363, 1038, 13184, 798... 

In [17]:
# =====================================
# Save cell and gene metadata
# =====================================
import scanpy as sc
import pandas as pd

# Reload the AnnData object
adata = sc.read_h5ad(h5ad_file)

print("\n==============================")
print("📊 Final AnnData Metadata Check")
print("==============================")
print(f"Cells: {adata.n_obs}, Genes: {adata.n_vars}\n")

# Cell metadata
obs_path = f"{prefix}.cells_metadata.csv"
adata.obs.to_csv(obs_path)
print(f"✅ Saved cell metadata (.obs) → {obs_path}")

# Gene/feature metadata
var_path = f"{prefix}.genes_metadata.csv"
adata.var.to_csv(var_path)
print(f"✅ Saved gene metadata (.var) → {var_path}")

# Also preview in console
print("\n• .obs (cell metadata) columns:")
print(list(adata.obs.columns))
print(adata.obs.head())

print("\n• .var (gene metadata) columns:")
print(list(adata.var.columns))
print(adata.var.head())



📊 Final AnnData Metadata Check
Cells: 104438, Genes: 21862

✅ Saved cell metadata (.obs) → hypoxia_all_samples_tokens.cells_metadata.csv
✅ Saved gene metadata (.var) → hypoxia_all_samples_tokens.genes_metadata.csv

• .obs (cell metadata) columns:
['sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'condition', '_scvi_batch', '_scvi_labels', 'leiden', 'merged_labels', 'class', 'broad_class', 'n_counts', 'counts', 'cell_id']
                       sample  n_genes_by_counts  log1p_n_genes_by_counts  \
TCAAGACCACAAAGCG-1  AP4765_03              12290                 9.416623   
CTCCACATCGCC

In [ ]:
# 1. hypoxia_all_samples_tokens.all.csv (~1.46M rows)
# This is the full token dataset exported from HuggingFace datasets.
# Each cell is represented not as one row, but as a sequence of tokens (gene IDs mapped to token IDs).
# If each of your ~104,439 cells gets ~14 tokens (on average), that already gives ~1.46 million rows in the .all.csv.
# This is expected, because the dataset is at the token level, not the cell level.

# 2. hypoxia_all_samples_tokens.cells_metadata.csv (104,439 rows)
# This matches exactly your number of cells.
# Each row = one cell.

# Columns = .obs metadata (sample, class, broad_class, condition, n_counts, etc.).
# ✅ This is the file you’d usually join with embeddings or labels.

# 3. hypoxia_all_samples_tokens.genes_metadata.csv (21,863 rows)
# This matches your number of genes (adata.n_vars).
# Each row = one gene.

# Columns = .var metadata (gene IDs, gene symbols, flags like mt/ribo/hb, etc.).
# ✅ This is what you’d use to check features (genes).

# ⚖️ Why the mismatch?
# Cells metadata file = one row per cell (104,439).
# Genes metadata file = one row per gene (21,863).
# Tokens file = one row per token, so it scales with both cells × expressed genes per cell. 
# That’s why it explodes in size.

In [ ]:
#!/usr/bin/env bash
# Usage:
#   ./extract_seqs.sh input.txt > sequences.txt
#   # or: cat input.txt | ./extract_seqs.sh

# Split fields at either "]," or "," and print the token right after "],"
# awk -F'],|,' '
#  NF >= 2 {
#    print $2
#  }
# ' "$@"

# ./script_extract_seqs.sh hypoxia_all_samples_tokens.all.csv | sort -u | wc -l